# Evidencia de aprendizaje 3 - Grupo 2 - Comparativa climatica entre Medellin y Madrid
### Fuentes de datos utilizadas

**Medellín y Madrid:**
   - **Fuente:** Open-Meteo Historical Weather API.
   - **Enlace:** https://open-meteo.com/
   - **Variables utilizadas:** `date` (Fecha), `daily_max_temperature` (Temperatura maxima diaria °C), `daily_precipitation_sum` (Precipitación diaria en mm).
   - **Periodo:** 2023-01-01 a 2023-12-31 (Año completo 2023).
   - **Frecuencia:** Diaria.

In [1]:
import os
import json
import urllib.request
import pandas as pd

def datos_clima(latitud, longitud, ciudad_nombre, fecha_inicio="2023-01-01", fecha_fin="2023-12-31"):
 
    os.makedirs("datos", exist_ok=True)
    filepath = f"datos/{ciudad_nombre.lower()}.csv"
    
    url = (
        f"https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={latitud}&longitude={longitud}&"
        f"start_date={fecha_inicio}&end_date={fecha_fin}&"
        f"daily=temperature_2m_max,temperature_2m_min,precipitation_sum&timezone=auto"
    )
    
    print(f"Conectando con Open-Meteo para {ciudad_nombre}...")

    peticion_datos = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(peticion_datos) as response:
        datos_json = json.loads(response.read().decode())
    
    df = pd.DataFrame(datos_json['daily'])
    
    df = df.rename(columns={
        'time': 'date',
        'temperature_2m_max': 'daily_max_temperature',
        'temperature_2m_min': 'daily_min_temperature',
        'precipitation_sum': 'daily_precipitation_sum'
    })
    
    df.to_csv(filepath, index=False)
    print(f"-> Descarga exitosa. Archivo generado: {filepath} ({len(df)} registros)")
    
    return df

# Se utilizan las coordenadas de Medellin y Madrid para obtener los datos limitando la ubicación de estas ciudades
df_medellin = datos_clima(6.2442, -75.5812, "medellin")
df_madrid = datos_clima(40.4168, -3.7038, "madrid")

print("\nPrimeras filas de Medellín:")
print(df_medellin.head(3))

Conectando con Open-Meteo para medellin...
-> Descarga exitosa. Archivo generado: datos/medellin.csv (365 registros)
Conectando con Open-Meteo para madrid...
-> Descarga exitosa. Archivo generado: datos/madrid.csv (365 registros)

Primeras filas de Medellín:
         date  daily_max_temperature  daily_min_temperature  \
0  2023-01-01                   25.1                   14.4   
1  2023-01-02                   25.2                   12.9   
2  2023-01-03                   23.2                   15.0   

   daily_precipitation_sum  
0                      1.3  
1                      1.3  
2                      8.3  


In [2]:
df_medellin = pd.read_csv("datos/medellin.csv")
df_madrid = pd.read_csv("datos/madrid.csv")

# Se convierten las fechas a tipo DateTime para mejorar su lectura
df_medellin['date'] = pd.to_datetime(df_medellin['date'])
df_madrid['date'] = pd.to_datetime(df_madrid['date'])

# Revisión y reporte
print("--- MEDELLÍN ---")
print(f"Dimensiones: {df_medellin.shape}")
print(f"Fecha inicial: {df_medellin['date'].min().strftime('%Y-%m-%d')} | Fecha final: {df_medellin['date'].max().strftime('%Y-%m-%d')}")
print(f"Valores nulos:\n{df_medellin.isna().sum().to_dict()}")
print(f"Fechas duplicadas: {df_medellin.duplicated(subset=['date']).sum()}")

print("\n--- MADRID ---")
print(f"Dimensiones: {df_madrid.shape}")
print(f"Fecha inicial: {df_madrid['date'].min().strftime('%Y-%m-%d')} | Fecha final: {df_madrid['date'].max().strftime('%Y-%m-%d')}")
print(f"Valores nulos:\n{df_madrid.isna().sum().to_dict()}")
print(f"Fechas duplicadas: {df_madrid.duplicated(subset=['date']).sum()}")

df_merged = pd.merge(
    df_medellin, df_madrid, 
    on='date', 
    suffixes=('_medellin', '_madrid'), 
    how='inner'
).dropna()

print(f"\nTotal de observaciones diarias para análisis: {len(df_merged)} días.")

--- MEDELLÍN ---
Dimensiones: (365, 4)
Fecha inicial: 2023-01-01 | Fecha final: 2023-12-31
Valores nulos:
{'date': 0, 'daily_max_temperature': 0, 'daily_min_temperature': 0, 'daily_precipitation_sum': 0}
Fechas duplicadas: 0

--- MADRID ---
Dimensiones: (365, 4)
Fecha inicial: 2023-01-01 | Fecha final: 2023-12-31
Valores nulos:
{'date': 0, 'daily_max_temperature': 0, 'daily_min_temperature': 0, 'daily_precipitation_sum': 0}
Fechas duplicadas: 0

Total de observaciones diarias para análisis: 365 días.


## Analisis utilizando NumPy

Se extraen los arreglos para aplicar operaciones vectoriales, slicing e inspección con booleanos.

In [4]:
import numpy as np
import pandas as pd

# 1. Conversión de las columnas a arrays de NumPy
t_max_med = df_merged['daily_max_temperature_medellin'].to_numpy()
t_min_med = df_merged['daily_min_temperature_medellin'].to_numpy()
p_med = df_merged['daily_precipitation_sum_medellin'].to_numpy()

t_max_mad = df_merged['daily_max_temperature_madrid'].to_numpy()
t_min_mad = df_merged['daily_min_temperature_madrid'].to_numpy()
p_mad = df_merged['daily_precipitation_sum_madrid'].to_numpy()

# Booleanos
# El umbral 1 se define como los días lluviosos, es decir aquellos con una precipitación mayor a 0.5mm (x > 0.5 mm)
UMBRAL_LLUVIA = 0.5
mask_lluvia_med = p_med > UMBRAL_LLUVIA
mask_lluvia_mad = p_mad > UMBRAL_LLUVIA

# El Umbral 2se define como los dias calurosos, aquellos en que la temperatura fue mayor a 28 grados (x > 28.0 °C)
UMBRAL_CALOR = 28.0
mask_calor_med = t_max_med > UMBRAL_CALOR
mask_calor_mad = t_max_med > UMBRAL_CALOR

# Se obtienen estadisticas descriptivas utilizando solo las funciones de NumPy
stats_df = pd.DataFrame({
    "Métrica": [
        "Temperatura Máxima Promedio (°C)",
        "Temperatura Mínima Promedio (°C)",
        "Temperatura Absoluta Mínima (°C)",
        "Temperatura Absoluta Máxima (°C)",
        "Precipitación Total Acumulada (mm)",
        "Número de Días Lluviosos (>0.5 mm)",
        f"Número de Días Calurosos (> {UMBRAL_CALOR}°C)"
    ],
    "Medellín": [
        np.mean(t_max_med),
        np.mean(t_min_med),
        np.min(t_min_med),
        np.max(t_max_med),
        np.sum(p_med),
        np.sum(mask_lluvia_med),
        np.sum(mask_calor_med)
    ],
    "Madrid": [
        np.mean(t_max_mad),
        np.mean(t_min_mad),
        np.min(t_min_mad),
        np.max(t_max_mad),
        np.sum(p_mad),
        np.sum(mask_lluvia_mad),
        np.sum(mask_calor_mad)
    ]
})

print("\n=== RESUMEN ESTADÍSTICO AMPLIA CON NUMPY ===")
print(stats_df.to_string(index=False))


=== RESUMEN ESTADÍSTICO AMPLIA CON NUMPY ===
                            Métrica    Medellín     Madrid
   Temperatura Máxima Promedio (°C)   26.602192  21.938904
   Temperatura Mínima Promedio (°C)   16.065479  10.097260
   Temperatura Absoluta Mínima (°C)   12.900000  -4.400000
   Temperatura Absoluta Máxima (°C)   30.400000  40.000000
 Precipitación Total Acumulada (mm) 1982.800000 516.700000
 Número de Días Lluviosos (>0.5 mm)  273.000000  77.000000
Número de Días Calurosos (> 28.0°C)   85.000000  85.000000
